# Convolutional Neural Networks (CNNs) with MNIST

This tutorial explains the core ideas behind a **Convolutional Neural Network (CNN)** by building a small one from scratch and training it to recognize handwritten digits (the MNIST dataset).

It is designed to run in **Google Colab**.
> Optional: In Colab, go to `Runtime > Change runtime type` and select **GPU** to train faster. A CPU also works fine for this small model.

## Learning objectives

By the end of this tutorial, you will understand:

1. How an image changes as it passes through a CNN
2. What convolution, ReLU, pooling, flattening, and fully connected layers do
3. How to divide data into training, validation, and test sets
4. How to train and save a CNN model
5. How to test the trained model
6. How to recognize overfitting

## 1. Import libraries and select CPU/GPU

We use **PyTorch** to build and train the network, **torchvision** to download MNIST, and **matplotlib** to visualize images.

A GPU can train the model faster, but this tutorial is small enough to also run comfortably on a CPU.

**New functions used below**

- `torch.device(name)` — creates an object representing where tensors/models will be stored and computed, e.g. `"cpu"` or `"cuda"`.
- `torch.cuda.is_available()` — returns `True` if a GPU is available to PyTorch, `False` otherwise.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, random_split

from torchvision import datasets, transforms

import numpy as np
import matplotlib.pyplot as plt

# A GPU makes training much faster, but everything here also works on a CPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 2. Set a random seed

Neural networks use random numbers to initialize their weights and to shuffle data. Setting a **seed** makes those "random" choices repeatable, so you get the same results every time you re-run the notebook. This is useful for learning and debugging.

**New functions used below**

- `torch.manual_seed(seed)` — fixes the random number generator PyTorch uses (for weight initialization, shuffling, etc.), so results are reproducible.
- `np.random.seed(seed)` — same idea, but for NumPy's random number generator.

In [ ]:
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

## 3. Load MNIST

MNIST is a dataset of 70,000 grayscale images of handwritten digits (0-9), each 28x28 pixels. `torchvision` can download it automatically.

`transforms.ToTensor()` converts each image into a PyTorch tensor and scales pixel values from `[0, 255]` to `[0.0, 1.0]`.

**New functions used below**

- `transforms.ToTensor()` — creates a transform that converts a PIL image into a PyTorch tensor and scales pixel values to `[0.0, 1.0]`.
- `datasets.MNIST(root, train, download, transform)` — downloads (if needed) and loads the MNIST dataset. `train=True` gives the training split, `train=False` gives the test split.

In [ ]:
transform = transforms.ToTensor()

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

print("Training set size:", len(train_dataset))
print("Test set size:", len(test_dataset))

## 4. Display images and inspect tensor shapes

Let's look at what an image actually is: a tensor with a shape, a data type, and a range of pixel values.

**New functions used below**

- `plt.subplots(nrows, ncols, figsize=(w, h))` — creates a figure and a grid of subplot axes, used to draw multiple images side by side.
- `axes[i].imshow(image, cmap="gray")` — draws an image on subplot `i`; `cmap="gray"` renders it in grayscale.
- `tensor.squeeze()` — removes dimensions of size 1 (e.g., turns a `[1, 28, 28]` image into `[28, 28]` so it can be plotted).
- `axes[i].set_title(text)` — sets the title shown above a subplot.
- `axes[i].axis("off")` — hides the axis ticks/labels for a cleaner image.
- `plt.show()` — renders the figure.

In [ ]:
image, label = train_dataset[0]

print("Image shape:", image.shape)   # [channels, height, width]
print("Image dtype:", image.dtype)
print("Pixel value range:", image.min().item(), "to", image.max().item())
print("Label:", label)

# Show the first 8 images with their labels
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for i in range(8):
    img, lbl = train_dataset[i]
    axes[i].imshow(img.squeeze(), cmap="gray")
    axes[i].set_title(str(lbl))
    axes[i].axis("off")
plt.show()

## 5. Batches, iterations, and epochs

Instead of feeding the whole dataset to the model at once, we split it into small groups called **batches**.

- **Batch size**: number of images processed together in one step.
- **Iteration**: one update of the model's weights, using one batch.
- **Epoch**: one full pass through the entire training set (i.e., all iterations needed to see every image once).

For example, with 60,000 training images and a batch size of 64, one epoch needs about `60000 / 64 ≈ 938` iterations.

In [ ]:
batch_size_demo = 64
num_batches_per_epoch = len(train_dataset) // batch_size_demo

print(f"Training images: {len(train_dataset)}")
print(f"Batch size: {batch_size_demo}")
print(f"Batches (iterations) per epoch: {num_batches_per_epoch}")

## 6. Define the CNN architecture

Our network follows this structure:

```
Input image
Shape: 1 x 28 x 28
        |
Convolution layer (1 -> 16 feature maps, 3x3 kernel, padding=1)
Output: 16 x 28 x 28
        |
ReLU activation
        |
Max pooling (2x2)
Output: 16 x 14 x 14
        |
Convolution layer (16 -> 32 feature maps, 3x3 kernel, padding=1)
Output: 32 x 14 x 14
        |
ReLU activation
        |
Max pooling (2x2)
Output: 32 x 7 x 7
        |
Flatten
32 x 7 x 7 = 1,568 values
        |
Fully connected layer (1,568 -> 128)
        |
ReLU activation
        |
Dropout
        |
Output layer (128 -> 10 classes)
```

**New functions/classes used below**

- `nn.Module` — the base class every PyTorch model inherits from; it tracks the model's layers and parameters.
- `nn.Conv2d(in_channels, out_channels, kernel_size, padding)` — creates a convolution layer.
- `nn.MaxPool2d(kernel_size, stride)` — creates a max pooling layer.
- `nn.Linear(in_features, out_features)` — creates a fully connected (dense) layer.
- `nn.Dropout(p)` — creates a dropout layer that randomly zeroes out a fraction `p` of values during training.
- `F.relu(x)` — applies the ReLU activation function to a tensor.
- `torch.flatten(x, start_dim)` — flattens a tensor into a longer 1D vector, starting at dimension `start_dim` (keeping the batch dimension, dimension 0, intact).
- `module.to(device)` — moves a model's (or tensor's) data to the specified device (CPU or GPU).
- `model.parameters()` — returns an iterator over all trainable weights in the model.
- `tensor.numel()` — returns the total number of elements in a tensor.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # Convolution layer 1: 1 input channel -> 16 feature maps
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        # Convolution layer 2: 16 input channels -> 32 feature maps
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)

        # Max pooling halves the height and width each time it's used
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Fully connected (dense) layers
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

        self.dropout = nn.Dropout(0.25)

    def forward(self, x):
        x = self.conv1(x)          # -> 16 x 28 x 28
        x = F.relu(x)
        x = self.pool(x)           # -> 16 x 14 x 14

        x = self.conv2(x)          # -> 32 x 14 x 14
        x = F.relu(x)
        x = self.pool(x)           # -> 32 x 7 x 7

        x = torch.flatten(x, 1)    # -> 1,568 values
        x = self.fc1(x)            # -> 128
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)            # -> 10 (logits, one score per digit)
        return x


model = SimpleCNN().to(device)
print(model)

num_params = sum(p.numel() for p in model.parameters())
print("\nTotal trainable parameters:", num_params)

### Bonus: plot the model structure

Two simple ways to visualize the network in Colab:

1. **`torchinfo.summary()`** — prints a clean table with every layer, its output shape, and parameter count.
2. **A block diagram** — draws a simple box-and-arrow diagram of just the layers (no low-level math operations), matching the architecture diagram from section 6.

**New functions used below**

- `torchinfo.summary(model, input_size)` — builds and prints a layer-by-layer summary table for a given input shape.
- `graphviz.Digraph()` — creates an empty diagram that we can add boxes and arrows to.
- `dot.node(name, label, shape)` — adds one box to the diagram.
- `dot.edge(from_name, to_name)` — draws an arrow connecting two boxes.

In [ ]:
# 1. Text summary: layer-by-layer table with output shapes and parameter counts
!pip install torchinfo -q

from torchinfo import summary

summary(model, input_size=(1, 1, 28, 28))

In [ ]:
# 2. Block diagram: just the layers, as simple boxes and arrows
import graphviz

layers = [
    "Input\n1 x 28 x 28",
    "conv1 (Conv2d)\n-> 16 x 28 x 28",
    "ReLU",
    "pool (MaxPool2d)\n-> 16 x 14 x 14",
    "conv2 (Conv2d)\n-> 32 x 14 x 14",
    "ReLU",
    "pool (MaxPool2d)\n-> 32 x 7 x 7",
    "Flatten\n-> 1,568",
    "fc1 (Linear)\n-> 128",
    "ReLU",
    "Dropout",
    "fc2 (Linear)\n-> 10",
]

dot = graphviz.Digraph(format="png")
dot.attr(rankdir="TB")

for i, layer in enumerate(layers):
    dot.node(str(i), layer, shape="box")

for i in range(len(layers) - 1):
    dot.edge(str(i), str(i + 1))

dot

## 7. Explain each layer

- **Convolution layer**: slides small filters (3x3 "kernels") across the image to detect simple patterns like edges and curves. Each filter produces one *feature map*. `padding=1` keeps the output the same height/width as the input.
- **ReLU activation**: replaces every negative value with 0 (`max(0, x)`). This lets the network learn non-linear patterns instead of only straight-line relationships.
- **Max pooling**: looks at small 2x2 windows and keeps only the largest value, shrinking the feature maps and keeping the strongest signals. This makes the network faster and more tolerant to small shifts in the image.
- **Flatten**: reshapes the multi-dimensional feature maps into a single long vector so it can be fed into a fully connected layer.
- **Fully connected (dense) layer**: connects every input value to every output neuron, combining all the features to make a decision.
- **Dropout**: randomly "turns off" a fraction of neurons during training only. This prevents the network from relying too heavily on any single neuron and helps reduce overfitting.
- **Output layer**: produces 10 raw scores ("logits"), one for each digit class (0-9).

## 8. Pass one image through every layer

Let's take a single image and manually push it through each layer, printing the shape after every step. Compare these shapes with the architecture diagram above — they should match exactly.

**New functions used below**

- `tensor.unsqueeze(dim)` — inserts a new dimension of size 1 at position `dim` (here, adding a batch dimension).
- `torch.no_grad()` — a context manager that disables gradient tracking, used when we only want to run the model forward without training it (saves memory and computation).

In [ ]:
sample_image, sample_label = train_dataset[0]
x = sample_image.unsqueeze(0).to(device)   # add a batch dimension -> [1, 1, 28, 28]
print("Input:               ", x.shape)

with torch.no_grad():
    x1 = model.conv1(x)
    print("After conv1:        ", x1.shape)

    x2 = F.relu(x1)
    print("After ReLU:         ", x2.shape)

    x3 = model.pool(x2)
    print("After max pooling:  ", x3.shape)

    x4 = model.conv2(x3)
    print("After conv2:        ", x4.shape)

    x5 = F.relu(x4)
    print("After ReLU:         ", x5.shape)

    x6 = model.pool(x5)
    print("After max pooling:  ", x6.shape)

    x7 = torch.flatten(x6, 1)
    print("After flatten:      ", x7.shape)

    x8 = model.fc1(x7)
    print("After fc1:          ", x8.shape)

    x9 = F.relu(x8)
    x10 = model.dropout(x9)
    print("After dropout:      ", x10.shape)

    x11 = model.fc2(x10)
    print("After output layer: ", x11.shape)

## 9. Visualize feature maps and pooling

Each feature map highlights a different pattern (e.g., edges in different directions). Notice how pooling shrinks the maps but keeps their most important features.

**New functions used below**

- `tensor.cpu()` — moves a tensor from the GPU back to the CPU (needed before plotting with matplotlib).
- `fig.suptitle(text)` — adds a title above an entire figure (as opposed to a single subplot).

In [ ]:
def show_feature_maps(feature_maps, title, max_maps=8):
    feature_maps = feature_maps.squeeze(0).cpu()  # remove the batch dimension
    num_maps = min(max_maps, feature_maps.shape[0])
    fig, axes = plt.subplots(1, num_maps, figsize=(num_maps * 1.5, 2))
    fig.suptitle(title)
    for i in range(num_maps):
        axes[i].imshow(feature_maps[i], cmap="gray")
        axes[i].axis("off")
    plt.show()

show_feature_maps(x2, "After conv1 + ReLU (16 channels, 28x28)")
show_feature_maps(x3, "After max pooling (16 channels, 14x14)")
show_feature_maps(x5, "After conv2 + ReLU (first 8 of 32 channels, 14x14)")
show_feature_maps(x6, "After max pooling (first 8 of 32 channels, 7x7)")

## 10. Explain logits and probabilities

The output layer produces 10 raw numbers called **logits**. They can be any value (positive or negative) and don't directly mean "probability".

**Softmax** converts logits into probabilities that all add up to 1, so we can interpret them as "how confident the model is" for each digit.

Since our model hasn't been trained yet, the probabilities below should look almost random (close to 10% for every digit).

**New functions used below**

- `F.softmax(x, dim)` — converts raw scores (logits) into probabilities that sum to 1 along dimension `dim`.
- `torch.argmax(x)` — returns the index of the largest value in a tensor (here, the predicted digit).
- `tensor.detach()` — returns a copy of a tensor that is disconnected from gradient tracking, so it can be safely converted to NumPy/plotted.
- `tensor.numpy()` — converts a CPU tensor into a NumPy array.
- `plt.bar(x, heights)` — draws a bar chart.
- `plt.xticks(values)` — sets the tick marks shown on the x-axis.

In [ ]:
logits = x11.squeeze(0).cpu()
probabilities = F.softmax(logits, dim=0)

print("Logits (raw scores):\n", logits.numpy())
print("\nProbabilities (sum to 1):\n", probabilities.numpy())
print("\nPredicted digit:", torch.argmax(probabilities).item(), " | True label:", sample_label)

plt.bar(range(10), probabilities.detach().numpy())
plt.xlabel("Digit class")
plt.ylabel("Probability")
plt.title("Predicted probabilities (untrained model)")
plt.xticks(range(10))
plt.show()

## 11. Explain forward pass, loss, backpropagation, and optimizer

Training a network repeats four steps for every batch of images:

1. **Forward pass**: feed the images through the network to get predictions (logits).
2. **Loss**: measure how wrong the predictions are, compared to the true labels. We use `CrossEntropyLoss`, which is the standard loss for classification.
3. **Backpropagation**: compute how much each weight in the network contributed to the error (the gradients), using `loss.backward()`.
4. **Optimizer step**: nudge each weight slightly in the direction that reduces the loss, using `optimizer.step()`. We use the `Adam` optimizer.

Let's demonstrate this on a single small batch and see the loss go down after just one update.

**New functions used below**

- `nn.CrossEntropyLoss()` — creates the standard loss function for multi-class classification; it compares predicted logits to true labels.
- `optim.Adam(parameters, lr)` — creates the Adam optimizer, which updates model weights using the given learning rate `lr`.
- `torch.stack(list_of_tensors)` — combines a list of tensors into one tensor with a new batch dimension.
- `torch.tensor(data)` — creates a tensor from plain Python data (e.g., a list of numbers).
- `optimizer.zero_grad()` — clears old gradients before computing new ones.
- `loss.backward()` — runs backpropagation, computing gradients for every model weight.
- `optimizer.step()` — updates the model weights using the gradients computed by `backward()`.

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Grab one small batch just to demonstrate a single training step
demo_images = torch.stack([train_dataset[i][0] for i in range(32)]).to(device)
demo_labels = torch.tensor([train_dataset[i][1] for i in range(32)]).to(device)

# 1. Forward pass: compute predictions
outputs = model(demo_images)
loss = loss_fn(outputs, demo_labels)
print("Loss before update:", loss.item())

# 2. Backward pass: compute gradients
optimizer.zero_grad()
loss.backward()

# 3. Optimizer step: update the weights using the gradients
optimizer.step()

# Check the loss again on the same batch after one update
with torch.no_grad():
    outputs_after = model(demo_images)
    loss_after = loss_fn(outputs_after, demo_labels)
print("Loss after one update:", loss_after.item())

## 12. Split training, validation, and test sets

- **Training set**: used to update the model's weights.
- **Validation set**: held out during training, used only to check how well the model generalizes to unseen data after each epoch. This helps us detect overfitting and pick the best model.
- **Test set**: used only once, at the very end, to report the model's final performance on completely untouched data.

MNIST already gives us a separate test set. We'll carve out 10% of the training set to use as our validation set.

**New functions used below**

- `random_split(dataset, lengths, generator)` — randomly splits a dataset into multiple smaller datasets of the given sizes.
- `torch.Generator().manual_seed(seed)` — creates a random number generator with a fixed seed, so the split is reproducible.

In [ ]:
val_fraction = 0.1
val_size = int(len(train_dataset) * val_fraction)
train_size = len(train_dataset) - val_size

train_subset, val_subset = random_split(
    train_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(SEED)
)

print("Training samples:  ", len(train_subset))
print("Validation samples:", len(val_subset))
print("Test samples:      ", len(test_dataset))

## 13. Create DataLoaders

A `DataLoader` automatically splits a dataset into batches and can shuffle the data each epoch. We shuffle only the training set, since validation and test sets don't need to be shuffled.

**New functions used below**

- `DataLoader(dataset, batch_size, shuffle)` — wraps a dataset so it can be iterated over in batches; `shuffle=True` shuffles the order every epoch.

In [ ]:
batch_size = 64

train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("Batches per epoch (training):", len(train_loader))

## 14. Train the model & 15. Validate after each epoch

We'll define two helper functions:

- `train_one_epoch`: runs the model in **training mode**, updating its weights on the training set.
- `evaluate`: runs the model in **evaluation mode** (no weight updates), used for both the validation and test sets.

After each epoch, we check the validation loss. This tells us how well the model is generalizing, since the validation set was never used to update the weights.

**New functions used below**

- `model.train()` — switches the model into training mode (e.g., enables dropout).
- `model.eval()` — switches the model into evaluation mode (e.g., disables dropout).
- `tensor.item()` — extracts a single Python number out of a tensor that contains only one value.
- `tensor.size(dim)` — returns the size of a tensor along dimension `dim`.
- `tensor.sum()` — adds up all the values in a tensor.

In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()
    total_loss = 0.0
    correct = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(dim=1) == labels).sum().item()

    avg_loss = total_loss / len(loader.dataset)
    accuracy = correct / len(loader.dataset)
    return avg_loss, accuracy


def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss = 0.0
    correct = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = loss_fn(outputs, labels)

            total_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(dim=1) == labels).sum().item()

    avg_loss = total_loss / len(loader.dataset)
    accuracy = correct / len(loader.dataset)
    return avg_loss, accuracy

**New functions used below**

- `model.state_dict()` — returns a dictionary containing all of the model's current weights.
- `torch.save(object, path)` — saves a Python object (here, the weights dictionary) to a file.

In [ ]:
# Start from a fresh, untrained model for the real training run
model = SimpleCNN().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 8
best_val_loss = float("inf")
checkpoint_path = "best_model.pth"

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

for epoch in range(1, num_epochs + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, loss_fn, optimizer)
    val_loss, val_acc = evaluate(model, val_loader, loss_fn)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch}/{num_epochs} | "
          f"train loss: {train_loss:.4f}, train acc: {train_acc:.4f} | "
          f"val loss: {val_loss:.4f}, val acc: {val_acc:.4f}")

    # Save the model only when validation loss improves
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), checkpoint_path)
        print(f"  -> New best model saved (val loss: {val_loss:.4f})")

## 16. Plot learning curves

Plotting the training vs. validation loss/accuracy over time helps us see how training is going:
- If both curves improve together, the model is learning well.
- If training loss keeps improving but validation loss gets worse, that's a sign of **overfitting** (more on this in section 21).

**New functions used below**

- `axes[i].plot(x, y, label=...)` — draws a line chart on a subplot.
- `axes[i].set_xlabel(text)` / `axes[i].set_ylabel(text)` — labels the x-axis / y-axis of a subplot.
- `axes[i].legend()` — displays a legend using the `label=` values passed to `plot()`.

In [ ]:
epochs_range = range(1, num_epochs + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs_range, history["train_loss"], label="Train loss")
axes[0].plot(epochs_range, history["val_loss"], label="Validation loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss curves")
axes[0].legend()

axes[1].plot(epochs_range, history["train_acc"], label="Train accuracy")
axes[1].plot(epochs_range, history["val_acc"], label="Validation accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Accuracy curves")
axes[1].legend()

plt.show()

## 17. Save the best model

We already saved the model's weights to `best_model.pth` during training, every time the validation loss improved. This means we keep the version of the model that generalized best, not necessarily the one from the very last epoch.

Let's confirm the checkpoint file exists.

**New functions used below**

- `os.path.exists(path)` — returns `True` if a file or folder exists at `path`.
- `os.path.getsize(path)` — returns the size of a file in bytes.

In [ ]:
import os

print("Checkpoint file exists:", os.path.exists(checkpoint_path))
print("File size (KB):", round(os.path.getsize(checkpoint_path) / 1024, 1))

## 18. Load the saved model

To use the best model (for testing, or later in an application), we create a fresh model instance and load the saved weights into it.

**New functions used below**

- `torch.load(path, map_location)` — loads a saved object (here, a weights dictionary) from disk; `map_location` ensures it loads onto the correct device.
- `model.load_state_dict(state_dict)` — copies the loaded weights into a model instance.

In [ ]:
loaded_model = SimpleCNN().to(device)
loaded_model.load_state_dict(torch.load(checkpoint_path, map_location=device))
loaded_model.eval()
print("Loaded the best model from", checkpoint_path)

## 19. Test on the untouched test set

This is the first and only time we use the test set. It gives us an honest estimate of how the model would perform on brand-new, real-world digits.

In [ ]:
test_loss, test_acc = evaluate(loaded_model, test_loader, loss_fn)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

## 20. Examine incorrect predictions

Looking at the mistakes the model makes is a great way to understand its limitations — often the misclassified digits are genuinely ambiguous or messily written.

**New functions used below**

- `list.extend(other_list)` — adds all items from `other_list` onto the end of `list`.
- Boolean tensor indexing, e.g. `images[mismatch]` — selects only the rows of `images` where `mismatch` is `True`, using a tensor of `True`/`False` values as a mask.

In [ ]:
loaded_model.eval()
wrong_images, wrong_preds, wrong_labels = [], [], []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = loaded_model(images)
        preds = outputs.argmax(dim=1)

        mismatch = preds != labels
        wrong_images.extend(images[mismatch].cpu())
        wrong_preds.extend(preds[mismatch].cpu())
        wrong_labels.extend(labels[mismatch].cpu())

        if len(wrong_images) >= 8:
            break

print("Number of incorrect predictions found (first batch scanned):", len(wrong_images))

fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for i in range(8):
    axes[i].imshow(wrong_images[i].squeeze(), cmap="gray")
    axes[i].set_title(f"Pred: {wrong_preds[i].item()}\nTrue: {wrong_labels[i].item()}")
    axes[i].axis("off")
plt.tight_layout()
plt.show()

## 21. Demonstrate underfitting and overfitting

- **Underfitting**: the model is too simple or undertrained. Both training and validation loss stay high, because the model hasn't learned enough patterns.
- **Overfitting**: the model memorizes the training data instead of learning general patterns. Training loss keeps dropping, but validation loss stops improving or gets worse.

To clearly see overfitting, we train a fresh model on a tiny subset of only 200 images for many epochs. With so little data, the model can "memorize" the training images, and the gap between training and validation loss should grow.

In [ ]:
# Use a very small training subset so the model can "memorize" it quickly
small_train_subset, _ = random_split(
    train_subset, [200, len(train_subset) - 200], generator=torch.Generator().manual_seed(SEED)
)
small_train_loader = DataLoader(small_train_subset, batch_size=32, shuffle=True)

overfit_model = SimpleCNN().to(device)
overfit_optimizer = optim.Adam(overfit_model.parameters(), lr=0.001)

overfit_history = {"train_loss": [], "val_loss": []}
overfit_epochs = 25

for epoch in range(1, overfit_epochs + 1):
    train_loss, _ = train_one_epoch(overfit_model, small_train_loader, loss_fn, overfit_optimizer)
    val_loss, _ = evaluate(overfit_model, val_loader, loss_fn)
    overfit_history["train_loss"].append(train_loss)
    overfit_history["val_loss"].append(val_loss)

plt.plot(range(1, overfit_epochs + 1), overfit_history["train_loss"], label="Train loss (200 images)")
plt.plot(range(1, overfit_epochs + 1), overfit_history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Overfitting demo: training on only 200 images")
plt.legend()
plt.show()

**How to read the chart above:** the training loss (on only 200 images) drops close to zero, because the model memorizes them. The validation loss stays much higher and may even increase — this gap between training and validation loss is the signature of overfitting.

Common ways to reduce overfitting: use more training data, add dropout (like we did), stop training earlier (early stopping), or use a simpler model.

## Summary

You've now seen the full lifecycle of a CNN:

1. An image (`1x28x28`) flows through convolution, ReLU, and pooling layers, shrinking spatially while growing in depth (`16x28x28` -> `16x14x14` -> `32x14x14` -> `32x7x7`).
2. It's flattened and passed through fully connected layers to produce 10 logits, turned into probabilities with softmax.
3. Data is split into training, validation, and test sets to fairly measure learning and generalization.
4. The model is trained with forward passes, loss, backpropagation, and an optimizer, while the best-performing version is checkpointed to disk.
5. The saved model is reloaded and evaluated once on the test set for an honest final score.
6. Comparing training vs. validation loss reveals underfitting and overfitting.

**Next steps to try:** train for more epochs, add a third convolution layer, try data augmentation, or experiment with different dropout rates.